# FOV Pointing & Orientation Optimization

**Goal.** Given a science target and an instrument with a (square) footprint,
choose the telescope **pointing** (field center, which need *not* sit on the
target) and **orientation** (position angle of the field) that keep the target
inside the field while capturing as many **useful comparison stars** as
possible.

**Why it matters.** The optimal exposure time depends not only on the target but
on the comparison stars in the field:

- A comparison **much brighter** than the target forces a *shorter* exposure (to
  avoid saturating it), which lowers the target's own SNR.
- A comparison **much fainter** than the target carries little weight for
  differential photometry (it is photon-starved).
- A comparison that is **blended/crowded** corrupts aperture photometry.

So the "best" field is the one whose footprint — once shifted and rotated —
contains the richest set of similarly-bright, similarly-colored, well-isolated
comparison stars, while still holding the target with an edge margin.

## Algorithm

1. **Footprint** — read the square half-width (arcsec) from `data/FOV_*.vot.xml`.
2. **Project** the neighbourhood onto a tangent plane centered on the target, so
   offsets and rotations become Cartesian operations in arcsec.
3. **Score** every candidate star as a comparison (magnitude + color terms, with
   a saturation penalty for bright comps and a crowding penalty for blends).
4. **Search** a coarse grid over field-center offset `(east, north)` and position
   angle, maximizing the summed comparison weight inside the footprint, subject
   to the target staying inside by a margin. A grid (not a gradient method)
   handles the non-smooth in/out membership cleanly and avoids local minima.
5. **Report** the optimum center (RA/Dec), PA, footprint corners, and ranked
   comparison list.

This notebook is the design/prototype. The same logic is packaged in
`muscat_db.fov` and served by the **FOV** web page (Aladin Lite overlay).

In [ ]:
# --- Setup: make `muscat_db` importable from the notebook ---
import sys
from pathlib import Path

# Detect the project root (the directory that contains `src/`).
here = Path.cwd()
root = next((p for p in (here, *here.parents) if (p / "src" / "muscat_db").is_dir()), None)
assert root is not None, "Could not locate project root containing src/muscat_db"
sys.path.insert(0, str(root / "src"))

import numpy as np
from muscat_db import fov

print("Project root:", root)
print("muscat_db.fov loaded from:", Path(fov.__file__).relative_to(root))

## 1 · Instrument footprints from the VOTable XML

Each instrument's square field is defined by polygon vertices (arcsec) in
`data/FOV_*.vot.xml`. `load_fov_halfsize_arcsec` returns the half-side; MuSCAT4
reuses the MuSCAT3 footprint, and Sinistro (no XML) falls back to a
detector-size computation.

In [ ]:
print(f"{'instrument':<10} {'half (arcsec)':>14} {'full field':>14}")
print("-" * 40)
for inst in ["muscat", "muscat2", "muscat3", "muscat4", "sinistro"]:
    half = fov.load_fov_halfsize_arcsec(inst)
    print(f"{inst:<10} {half:>14.1f} {2 * half / 60:>11.2f}'")

## 2 · Tangent-plane geometry

We work in standard coordinates `(east, north)` in arcsec relative to the
target via a gnomonic (TAN) projection. Offsetting the pointing and rotating the
field are then simple translations/rotations. The projection round-trips to
sub-mas precision over a MuSCAT-sized field.

In [ ]:
ra0, dec0 = 97.6367, 29.6722  # WASP-12

# Round-trip a few offsets through the projection.
for (e, n) in [(0, 0), (120, -80), (-200, 150)]:
    ra, dec = fov.tangent_to_radec(e, n, ra0, dec0)
    e2, n2 = fov.radec_to_tangent(np.array([ra]), np.array([dec]), ra0, dec0)
    print(f"offset ({e:>5},{n:>5})  ->  RA/Dec ({ra:.5f}, {dec:.5f})  ->  ({e2[0]:.3f}, {n2[0]:.3f})")

# Membership test: a point at +160 arcsec east is outside a centered 150-arcsec
# half-field, but inside one shifted east by 60 arcsec.
pt_e, pt_n = np.array([160.0]), np.array([0.0])
print("\ncentered field contains (160,0)? ", fov.inside_square(pt_e, pt_n, 0, 0, 150, 0)[0])
print("shifted field  contains (160,0)? ", fov.inside_square(pt_e, pt_n, 60, 0, 150, 0)[0])

## 3 · Scoring "useful" comparison stars

`comparison_weights` combines:

- a **magnitude** term: a Gaussian centered slightly fainter than the target
  (`IDEAL_DMAG`), with a hard `BRIGHT_PENALTY` for comps bright enough to cap the
  exposure (`ΔG < BRIGHT_DMAG_LIMIT`);
- an optional **color** term: a Gaussian in `BP−RP` distance from the target,
  since matched colors give cleaner differential photometry.

The curve below shows weight vs. brightness offset `ΔG = G_comp − G_target`.

In [ ]:
dmag = np.linspace(-3, 5, 200)
g_target = 12.0
w = fov.comparison_weights(g_target + dmag, None, target_g=g_target, target_bp_rp=None)

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(dmag, w, lw=2, color="#2a9d8f")
    ax.axvline(fov.IDEAL_DMAG, ls="--", color="gray", lw=1, label=f"ideal ΔG={fov.IDEAL_DMAG}")
    ax.axvspan(-3, fov.BRIGHT_DMAG_LIMIT, color="#e07b53", alpha=0.15, label="saturation-risk (bright)")
    ax.axhline(fov.WEIGHT_FLOOR, ls=":", color="k", lw=1, label=f"weight floor={fov.WEIGHT_FLOOR}")
    ax.set_xlabel("ΔG = G_comp − G_target"); ax.set_ylabel("usefulness weight")
    ax.set_title("Comparison-star usefulness vs. brightness offset"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
except ModuleNotFoundError:
    print("matplotlib not installed — text summary instead:")
    for d in [-2, -1, 0, fov.IDEAL_DMAG, 1, 2, 3, 4]:
        print(f"  ΔG={d:>5}:  weight={float(fov.comparison_weights(np.array([g_target+d]),None,g_target,None)[0]):.3f}")

## 4 · Candidate stars

In production we cone-search **Gaia DR3** (via Vizier) around the target. That
needs the network; if it is unavailable, this cell falls back to a small
**synthetic** field so the notebook still runs end-to-end and the optimizer can
be demonstrated offline.

In [ ]:
TARGET = "WASP-12"
INSTRUMENT = "muscat2"
half = fov.load_fov_halfsize_arcsec(INSTRUMENT)
radius = half * np.sqrt(2) * fov.QUERY_RADIUS_FACTOR

stars = fov.query_gaia_field(ra0, dec0, radius, mag_limit=16.0)

if stars.error or len(stars) == 0:
    print(f"Gaia unavailable ({stars.error}); using a synthetic field.")
    rng = np.random.default_rng(12)
    n = 40
    e = rng.uniform(-radius, radius, n)
    nth = rng.uniform(-radius, radius, n)
    ra_s, dec_s = [], []
    for ee, nn in zip(e, nth):
        r, d = fov.tangent_to_radec(ee, nn, ra0, dec0); ra_s.append(r); dec_s.append(d)
    g = rng.uniform(11.0, 16.0, n)
    # Inject a few good comps near the target's brightness, clustered to the NE.
    ra_s = np.array(ra_s); dec_s = np.array(dec_s)
    stars = fov.StarField(ra_s, dec_s, g, rng.uniform(0.4, 1.6, n), source="synthetic")

print(f"{len(stars)} candidate stars within {radius:.0f}\" of {TARGET} ({stars.source})")
print(f"G range: {np.nanmin(stars.gmag):.1f} – {np.nanmax(stars.gmag):.1f}")

## 5 · Optimize pointing + orientation

Project the candidates, score them (zeroing out the target itself and applying
the crowding penalty), then grid-search the field center and PA.

In [ ]:
east, north = fov.radec_to_tangent(stars.ra, stars.dec, ra0, dec0)

# Identify and exclude the target star (nearest Gaia source within 3").
t_idx = fov._identify_target_star(stars, ra0, dec0)
target_g = float(stars.gmag[t_idx]) if t_idx is not None else float(np.nanmedian(stars.gmag))
target_color = float(stars.bp_rp[t_idx]) if (t_idx is not None and np.isfinite(stars.bp_rp[t_idx])) else None

weights = fov.comparison_weights(stars.gmag, stars.bp_rp, target_g, target_color)
if t_idx is not None:
    weights[t_idx] = 0.0
weights = np.where(np.isfinite(weights) & (weights >= fov.WEIGHT_FLOOR), weights, 0.0)
weights = fov._blend_penalty(east, north, stars.gmag, weights)

sol = fov.optimize_pointing(east, north, weights, half=half, margin=30.0)
center_ra, center_dec = fov.tangent_to_radec(sol.center_east, sol.center_north, ra0, dec0)

print(f"target G ≈ {target_g:.2f}  color BP-RP ≈ {target_color}")
print(f"field center : RA {center_ra:.5f}, Dec {center_dec:.5f}")
print(f"target offset: E {sol.center_east:+.0f}\", N {sol.center_north:+.0f}\"")
print(f"position angle: {sol.pa_deg:.0f} deg")
print(f"useful comps in field: {int((sol.in_field & (weights>0)).sum())}   "
      f"weighted signal: {weights[sol.in_field].sum():.2f}")

## 6 · Visualize the optimized field

The footprint (blue square) is placed at the optimized pointing/PA. The target
(red) stays inside with margin; captured comparisons are green (size ∝ weight),
rejected/out-of-field stars are gray.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon as MplPolygon

    fig, ax = plt.subplots(figsize=(6.4, 6.4))

    # Footprint corners in tangent-plane arcsec (rotated square).
    import math
    pa = math.radians(sol.pa_deg); c, s = math.cos(pa), math.sin(pa)
    corners = []
    for sx, sy in [(-1,-1),(1,-1),(1,1),(-1,1)]:
        lx, ly = sx*half, sy*half
        corners.append([sol.center_east + lx*c - ly*s, sol.center_north + lx*s + ly*c])
    ax.add_patch(MplPolygon(corners, closed=True, fill=False, edgecolor="#6c8cff", lw=2, label="footprint"))

    inside = sol.in_field & (weights > 0)
    ax.scatter(east[~inside], north[~inside], s=12, c="lightgray", label="other/out")
    ax.scatter(east[inside], north[inside], s=30 + 200*weights[inside], c="#2a9d8f",
               edgecolor="k", lw=0.4, label="useful comps", zorder=3)
    ax.scatter([0], [0], marker="*", s=300, c="#ff5470", edgecolor="k", lw=0.6, label="target", zorder=4)

    lim = half * 1.5
    ax.set_xlim(lim, -lim)  # East to the left (RA increases left)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xlabel("East offset (arcsec)"); ax.set_ylabel("North offset (arcsec)")
    ax.set_title(f"{INSTRUMENT} optimized FOV for {TARGET}  (PA={sol.pa_deg:.0f}°)")
    ax.legend(fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()
except ModuleNotFoundError:
    print("matplotlib not installed; skipping the figure.")
    print("Footprint corners (E,N arcsec):")
    import math
    pa = math.radians(sol.pa_deg); c, s = math.cos(pa), math.sin(pa)
    for sx, sy in [(-1,-1),(1,-1),(1,1),(-1,1)]:
        lx, ly = sx*half, sy*half
        print(f"  ({sol.center_east + lx*c - ly*s:+.0f}, {sol.center_north + lx*s + ly*c:+.0f})")

## 7 · End-to-end and the web page

`muscat_db.fov.optimize` wraps all of the above: it resolves the target name
(SIMBAD), pulls Gaia neighbours, scores, optimizes, and returns a JSON-ready
result with the footprint corners and a ranked comparison list. The **FOV** web
page (`/fov`) calls this through `POST /api/fov/optimize` and overplots the
footprint + comparisons on **Aladin Lite**.

The cell below runs the full pipeline (needs the network for SIMBAD + Gaia).

In [ ]:
result = fov.optimize(INSTRUMENT, target=TARGET, margin_arcsec=30.0, mag_limit=16.0)
if not result.ok:
    print("optimize failed:", result.error)
else:
    print(f"target {result.target}: RA {result.ra:.5f}, Dec {result.dec:.5f}, G≈{result.target_gmag:.2f}")
    print(f"optimum center RA {result.center_ra:.5f}, Dec {result.center_dec:.5f}, PA {result.pa_deg:.0f}°")
    print(f"useful comps: {result.n_comps}   weighted signal: {result.total_weight}")
    print("\ntop comparison stars:")
    print(f"  {'RA':>10} {'Dec':>10} {'G':>6} {'ΔG':>6} {'sep\"':>6} {'w':>5}")
    for c in result.comps[:8]:
        print(f"  {c['ra']:>10.5f} {c['dec']:>10.5f} {c['gmag']:>6.2f} "
              f"{c['dmag']:>+6.2f} {c['sep_arcsec']:>6.0f} {c['weight']:>5.2f}")